# Chapter 3 pipeline tests

## Set Up

In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
#===
import sys
from pathlib import Path

In [2]:
project_root = Path().resolve().parent.parent
sys.path.append(str(project_root))
project_root

PosixPath('/Users/alejandrofp/Desktop/Projects/03_Flagship_Portfolio/job-intelligence-engine')

In [3]:
from src.job_intel.pipelines.chapter3_individual_positioning import main

## Run pipeline

In [4]:
main()

In [5]:
from pathlib import Path

OUT_DIR = Path("reports/chapter3")

## Recall outputs

In [6]:
import pandas as pd
jobs_df = pd.read_csv(OUT_DIR / "ranked_jobs.csv")
gaps_df = pd.read_csv(OUT_DIR / "skill_gaps.csv")

In [7]:
jobs_df[['job_id','suitability']]

,job_id,suitability
0,3025,0.941111
1,2870,0.937249
2,3033,0.928114
3,3130,0.927921
4,382,0.920601
5,3112,0.909705
6,2873,0.890127
7,2883,0.880910
8,2881,0.877632
9,3141,0.875663


In [8]:
gaps_df['skill_gap']

0     0.456435
1     0.423314
2     0.241627
3     0.241434
4     0.208891
5     0.204277
6     0.175809
7     0.140294
8     0.138739
9     0.137075
10    0.135120
11    0.130260
12    0.076680
13    0.049554
14    0.015000
15    0.014283
16    0.006073
17    0.004254
18    0.003895
19    0.001703
20    0.000007
21    0.000007
22    0.000003
23    0.000000
24    0.000000
25    0.000000
26    0.000000
Name: skill_gap, dtype: float64

## Tests

In [31]:
# === Chapter 3 Smoke Tests ===

# 1) Outputs exist and are non-empty
assert isinstance(jobs_df, pd.DataFrame)
assert isinstance(gaps_df, pd.DataFrame)

assert len(jobs_df) > 0, "No jobs returned"
assert len(gaps_df) > 0, "No skill gaps returned"

# 2) Required columns present in ranked jobs
required_job_cols = [
    "suitability",
    "skill_match_norm",
    "salary_score",
]
missing_job_cols = [c for c in required_job_cols if c not in jobs_df.columns]
assert not missing_job_cols, f"Missing job columns: {missing_job_cols}"

# 3) Suitability scores in valid range
assert jobs_df["suitability"].between(0, 1).all(), "Suitability outside [0,1]"
assert jobs_df["skill_match_norm"].between(0, 1).all(), "Skill match norm outside [0,1]"
assert jobs_df["salary_score"].between(0, 1).all(), "Salary score outside [0,1]"

# 4) Jobs sorted by suitability (descending)
assert jobs_df["suitability"].is_monotonic_decreasing, "Jobs not sorted by suitability"

# 5) Required columns present in gap table
required_gap_cols = ["skill", "job_skill_rate", "user_skill", "skill_gap"]
missing_gap_cols = [c for c in required_gap_cols if c not in gaps_df.columns]
assert not missing_gap_cols, f"Missing gap columns: {missing_gap_cols}"

# 6) Gap logic sanity
assert (gaps_df.loc[gaps_df["user_skill"] == 1, "skill_gap"] == 0).all(), \
    "User-owned skills should not have gaps"

assert gaps_df["skill_gap"].between(0, 1).all(), "Skill gaps outside [0,1]"

print("✅ Chapter 3 smoke tests passed")


✅ Chapter 3 smoke tests passed


## Extra tests

In [10]:
from src.job_intel.positioning import run_positioning
from src.job_intel.features.artefacts_ch3 import load_ch3_artefacts
from src.job_intel.features.skills_pca import SKILL_COLS
from src.job_intel.schemas import build_user_profile
from src.job_intel.features.skill_extractor import explain_matches

### Test entrypoint datasets

In [11]:
df, skill = load_ch3_artefacts()

In [12]:
print("TEST 1 - Job filter TEST")
print("Check whether the filter for jobs with zero skills was implemented correctly.")
print("---------------------------")

if sum(df[SKILL_COLS].sum(axis=1) == 0) == 0:
    print("✅ Filter applied correctly to artefacts: PASS")
else:
    print("❌ Filter applied correctly to artefacts: FAIL")

if (df["job_id"] == skill["job_id"]).mean() == 1:
    print("✅ df and skill matrix alignement: PASS")
else:
    print("❌ df and skill matrix alignement: FAIL")

TEST 1 - Job filter TEST
Check whether the filter for jobs with zero skills was implemented correctly.
---------------------------
✅ Filter applied correctly to artefacts: PASS
✅ df and skill matrix alignement: PASS


### Invariance tests
Create tests that assert **repeatability**: 
- Same user inputs + same artefacts ⇒ identical outputs 
- Re-running run_positioning() twice produces: 
    - identical candidates_df ordering 
    - identical suitability scores 
    - identical competitiveness scores 
    - identical skill gaps 
    - identical sensitivity summaries

In [13]:
random_job = 1034

skill_text = df.loc[df['job_id'] == random_job, 'Job Description'].iloc[0]
skill_text

'PDI is seeking a Data Engineer to join our team working in an exciting, high volume retail data ecosystem with over $100 billion of transaction log data. The candidate will benefit from a rich knowledge of software development practices and will need to apply those practices daily. Mastering the relationship between client data and our ecosystem will be an important element for this position.\n\nThis position plays a critical role at PDI and for both the retailer and their vendor partners to realize the most value from their data. The overarching responsibilities for this role are (i) on-boarding successfully and efficiently new clients, (ii) maintaining the data and ecosystem on an ongoing basis and introducing & implementing improvements as needed. The successful applicant will be joining an open and diverse team with a "Can Do" attitude, and a strong desire to make an impact in a start-up organization.\n\nRESPONSIBILITIES & TASKS\nOnboard new customers\' data into the PDI data ware

In [14]:
explain_matches(skill_text)

{'core_programming__basic': ['python', 'java', 'bash', 'linux', 'unix'],
 'core_programming__intermediate': [],
 'core_programming__advanced': [],
 'data_engineering_pipelines__basic': [],
 'data_engineering_pipelines__intermediate': [],
 'data_engineering_pipelines__advanced': [],
 'ml_ai__basic': [],
 'ml_ai__intermediate': [],
 'ml_ai__advanced': [],
 'analytics_stats__basic': ['math', 'integration'],
 'analytics_stats__intermediate': [],
 'analytics_stats__advanced': [],
 'bi_viz__basic': [],
 'bi_viz__intermediate': [],
 'bi_viz__advanced': [],
 'cloud__basic': ['cloud'],
 'cloud__intermediate': [],
 'cloud__advanced': [],
 'db_storage__basic': ['queries', 'sql'],
 'db_storage__intermediate': ['data warehouse'],
 'db_storage__advanced': [],
 'productivity_workflow__basic': [],
 'productivity_workflow__intermediate': [],
 'productivity_workflow__advanced': [],
 'soft_skills__core': ['problem solving',
  'communication skills',
  'communication',
  'collaborate',
  'team player',
  

In [15]:
profile = build_user_profile(skill_text=skill_text,
                            current_state='ALL',
                            job_title_family='data_scientist',
                            job_title_rich='ML_AI_data_scientist',
                            target_sectors=None,
                            salary_target=250000,
                            explain_skills=False)

In [16]:
run_sensitivity = True

profile_r1, jobs_df_r1, gaps_df_r1, sensitivity_r1 = run_positioning(skill_text=skill_text,
                                                                    current_state='ALL',
                                                                    job_title_family='data_scientist',
                                                                    job_title_rich='ML_AI_data_scientist',
                                                                    target_sectors=None,
                                                                    salary_target=250000,
                                                                    explain_skills=False,
                                                                    top_k_gaps = 50,
                                                                    return_top_n_jobs = 200,
                                                                    run_sensitivity=run_sensitivity
                                                                    )

profile_r2, jobs_df_r2, gaps_df_r2, sensitivity_r2 = run_positioning(skill_text=skill_text,
                                                                    current_state='ALL',
                                                                    job_title_family='data_scientist',
                                                                    job_title_rich='ML_AI_data_scientist',
                                                                    target_sectors=None,
                                                                    salary_target=250000,
                                                                    explain_skills=False,
                                                                    top_k_gaps = 50,
                                                                    return_top_n_jobs = 200,
                                                                    run_sensitivity=run_sensitivity
                                                                    )

In [17]:

print('TEST 2 - INVARIANCE TEST')
print('Run the pipeline twice with the same arguments and check for identical (expected) values.')
print('---------------------------')

if jobs_df_r1[['suitability','competitiveness_index']].isna().any().any() or jobs_df_r2[['suitability','competitiveness_index']].isna().any().any():
    print('❌ NA test: FAIL')
    raise ValueError('dfs have unexpected NAs')
else:
    print('✅ NA test: PASS')

if (jobs_df_r1['suitability'].round(5) == jobs_df_r2['suitability'].round(5)).mean() == 1:
    print('✅ Identical suitability scores: PASS')
else:
    max_diff = (jobs_df_r1['suitability'] - jobs_df_r2['suitability']).abs().max()
    print('❌ Identical suitability scores: FAIL')
    print(f'   max |Δ suitability| = {max_diff}')

if (jobs_df_r1['competitiveness_index'].round(5) == jobs_df_r2['competitiveness_index'].round(5)).mean() == 1:
    print('✅ Identical competitiveness scores: PASS')
else:
    max_diff = (jobs_df_r1['competitiveness_index'] - jobs_df_r2['competitiveness_index']).abs().max()
    print('❌ Identical competitiveness scores: FAIL')
    print(f'   max |Δ competitiveness| = {max_diff}')

if (gaps_df_r1['skill_gap'].round(5) == gaps_df_r2['skill_gap'].round(5)).mean() == 1:
    print('✅ Identical skill gap: PASS')
else:
    max_diff = (gaps_df_r1['skill_gap'] - gaps_df_r2['skill_gap']).abs().max()
    print('❌ Identical skill gap: FAIL')
    print(f'   max |Δ skill_gap| = {max_diff}')

if jobs_df_r1.shape == jobs_df_r2.shape:
    print('✅ Identical shape: PASS')
else:
    print('❌ Identical shape: FAIL')  

if jobs_df_r1['job_id'].reset_index(drop=True).equals(jobs_df_r2['job_id'].reset_index(drop=True)):
    print('✅ Positional equality: PASS')
else:
    print('❌ Positional equality: FAIL')      

if run_sensitivity:
    if (sensitivity_r1['suitability']['spearman_rho_vs_baseline'].round(5)
    == sensitivity_r2['suitability']['spearman_rho_vs_baseline'].round(5)).mean() == 1:
        print('✅ Identical suitability sensitivity: PASS')
    else:
        max_diff = (
        sensitivity_r1['suitability']['spearman_rho_vs_baseline']
        - sensitivity_r2['suitability']['spearman_rho_vs_baseline']
        ).abs().max()
        print('❌ Identical suitability sensitivity: FAIL')
        print(f'   max |Δ spearman_rho_vs_baseline| = {max_diff}')

    if (sensitivity_r1['competitiveness']['spearman_rho_vs_baseline'].round(5)
        == sensitivity_r2['competitiveness']['spearman_rho_vs_baseline'].round(5)).mean() == 1:
        print('✅ Identical competitiveness sensitivity: PASS')
    else:
        max_diff = (
            sensitivity_r1['competitiveness']['spearman_rho_vs_baseline']
            - sensitivity_r2['competitiveness']['spearman_rho_vs_baseline']
        ).abs().max()
        print('❌ Identical competitiveness sensitivity: FAIL')
        print(f'   max |Δ spearman_rho_vs_baseline| = {max_diff}')



TEST 2 - INVARIANCE TEST
Run the pipeline twice with the same arguments and check for identical (expected) values.
---------------------------
✅ NA test: PASS
✅ Identical suitability scores: PASS
✅ Identical competitiveness scores: PASS
✅ Identical skill gap: PASS
✅ Identical shape: PASS
✅ Positional equality: PASS
✅ Identical suitability sensitivity: PASS
✅ Identical competitiveness sensitivity: PASS


### Test candidate selection

#### Zero jobs

In [18]:
profile_zero, jobs_df_zero, gaps_df_zero, sensitivity_zero = run_positioning(skill_text=skill_text,
                                                                    current_state='TX',
                                                                    job_title_family='data_scientist',
                                                                    job_title_rich='ML_AI_data_scientist',
                                                                    target_sectors=["Real Estate",],
                                                                    salary_target=250000,
                                                                    explain_skills=False,
                                                                    top_k_gaps = 50,
                                                                    return_top_n_jobs = 200,
                                                                    run_sensitivity=run_sensitivity
                                                                    )

ValueError: No jobs available within the current constraints. Please widen your filters.

In [19]:
profile_1j, jobs_df_1j, gaps_df_1j, sensitivity_1j = run_positioning(skill_text=skill_text,
                                                                    current_state='ALL',
                                                                    job_title_family='data_scientist',
                                                                    job_title_rich='ML_AI_data_scientist',
                                                                    target_sectors=None,
                                                                    salary_target=250000,
                                                                    explain_skills=False,
                                                                    top_k_gaps = 1,
                                                                    return_top_n_jobs = 1,
                                                                    run_sensitivity=run_sensitivity
                                                                    )

In [20]:
jobs_df_1j

,job_id,Job Description,Rating,Size,Founded,Industry,Sector,role_source,state,ownership_clean,...,skill_PC9,skill_PC10,skill_match_score,skill_match_norm,salary_score,suitability,expected_missing,expected_missing_norm,salary_pct,competitiveness_index
2987,3006,Atlassian is continuing to hire with all inter...,4.4,1001 to 5000 employees,2002.0,Computer Hardware & Software,Information Technology,data_scientist,CA,public,...,0.162774,0.551393,0.559082,0.779541,0.9,0.815679,0.4855,0.017981,0.983871,0.500926


In [21]:
profile_1j, jobs_df_1j, gaps_df_1j, sensitivity_1j = run_positioning(skill_text='',
                                                                    current_state='ALL',
                                                                    job_title_family='data_scientist',
                                                                    job_title_rich='ML_AI_data_scientist',
                                                                    target_sectors=None,
                                                                    salary_target=250000,
                                                                    explain_skills=False,
                                                                    top_k_gaps = 50,
                                                                    return_top_n_jobs = 200,
                                                                    run_sensitivity=run_sensitivity
                                                                    )

In [22]:
jobs_df_1j

,job_id,Job Description,Rating,Size,Founded,Industry,Sector,role_source,state,ownership_clean,...,skill_PC9,skill_PC10,skill_match_score,skill_match_norm,salary_score,suitability,expected_missing,expected_missing_norm,salary_pct,competitiveness_index
1339,1346,Job Description\nRole Description\n\nHands on ...,4.1,51 to 200 employees,1999.0,Enterprise Software & Network Solutions,Information Technology,data_scientist,AZ,private,...,-0.760708,-0.167597,0.799598,0.899799,0.512,0.783459,0.234609,0.008689,0.451613,0.230151
197,200,This isÂPriyaÂfromÂComTech Global Inc.ÂI was r...,4.0,51 to 200 employees,NaN,"Health, Beauty, & Fitness",Consumer Services,data_scientist,NY,private,...,-0.188254,-0.040197,0.688609,0.844305,0.532,0.750613,11.751799,0.435252,0.540323,0.487787
146,148,Company: AI/Data Science\nLocation: New York C...,5.0,1 to 50 employees,1987.0,Staffing & Outsourcing,Business Services,data_scientist,NY,private,...,0.500130,0.260356,0.548131,0.774065,0.678,0.745246,0.534075,0.019781,0.838710,0.429245
3129,3149,", 2019\nWeekly Hours: 40\nRole Number:\n200058...",4.1,10000+ employees,1976.0,Computer Hardware & Software,Information Technology,data_scientist,CA,public,...,0.504084,-0.123519,0.517754,0.758877,0.626,0.719014,0.517600,0.019170,0.766129,0.392650
2944,2963,"Data Scientist, ML\nWe are an industry leading...",4.2,201 to 500 employees,1999.0,Staffing & Outsourcing,Business Services,data_scientist,CA,public,...,-0.673207,0.713337,0.256226,0.628113,0.774,0.671879,1.095342,0.040568,0.935484,0.488026
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
765,771,Passionate about precision medicine and advanc...,3.3,51 to 200 employees,1999.0,Financial Transaction Processing,Finance,data_scientist,IL,private,...,0.419174,0.909839,-0.565597,0.217201,0.392,0.269641,2.369360,0.087754,0.250000,0.168877
900,907,Passionate about precision medicine and advanc...,3.2,501 to 1000 employees,2015.0,Biotech & Pharmaceuticals,Biotech & Pharmaceuticals,data_scientist,IL,private,...,0.419174,0.909839,-0.565597,0.217201,0.342,0.254641,2.406247,0.089120,0.185484,0.137302
895,902,Passionate about precision medicine and advanc...,3.3,51 to 200 employees,1999.0,Financial Transaction Processing,Finance,data_scientist,IL,private,...,0.419174,0.909839,-0.565597,0.217201,0.342,0.254641,2.374899,0.087959,0.185484,0.136722
1694,1703,Advance your career and our mission\n\nAs a Se...,3.2,10000+ employees,1975.0,Investment Banking & Asset Management,Finance,data_scientist,PA,private,...,-0.081185,-0.154060,-0.517370,0.241315,0.248,0.243321,1.635735,0.060583,0.032258,0.046420


In [23]:
from src.job_intel.features.candidate_selection import candidate_set_construction    
profile, candidates_df = candidate_set_construction(
        df=jobs_df,
        skill_text=skill_text,
        current_state='ALL',
        job_title_family='data_scientist',
        job_title_rich=None,
        target_sectors=None,
        salary_target=250000,
        explain_skills=False,
    )

from src.job_intel.features.candidate_competitiveness import add_competitiveness
candidates_df = add_competitiveness(
        profile, candidates_df, skill.drop('productivity_workflow__intermediate_prob', axis = 1), use_rarity=True
    )

KeyError: "['productivity_workflow__intermediate_prob'] not in index"

In [24]:
x = 5
bad_ids = candidates_df["job_id"].head(x)

skill_prob_mismatch = skill[~skill["job_id"].isin(bad_ids)].copy()

_ = add_competitiveness(profile, candidates_df, skill_prob_mismatch, use_rarity=True)


KeyError: 'job_id mismatch: 5 candidate job_ids missing from skill_prob_matrix. Example: [3025, 2870, 3033, 3130, 382]'

### Behaviour check


#### suitability - increase w_skill

In [25]:
profile_r1, jobs_df_r1, gaps_df_r1, sensitivity_r1 = run_positioning(skill_text=skill_text,
                                                                    current_state='ALL',
                                                                    job_title_family='data_scientist',
                                                                    job_title_rich='ML_AI_data_scientist',
                                                                    target_sectors=None,
                                                                    w_skill= 0.7,
                                                                    salary_target=250000,
                                                                    explain_skills=False,
                                                                    top_k_gaps = 50,
                                                                    return_top_n_jobs = 200,
                                                                    run_sensitivity=False
                                                                    )

profile_r2, jobs_df_r2, gaps_df_r2, sensitivity_r2 = run_positioning(skill_text=skill_text,
                                                                    current_state='ALL',
                                                                    job_title_family='data_scientist',
                                                                    job_title_rich='ML_AI_data_scientist',
                                                                    target_sectors=None,
                                                                    w_skill= 0.9,
                                                                    salary_target=250000,
                                                                    explain_skills=False,
                                                                    top_k_gaps = 50,
                                                                    return_top_n_jobs = 200,
                                                                    run_sensitivity=False
                                                                    )

d1 = jobs_df_r1[["job_id","suitability","skill_match_norm"]].set_index("job_id")
d2 = jobs_df_r2[["job_id","suitability"]].set_index("job_id")

common = d1.index.intersection(d2.index)
d1 = d1.loc[common]
d2 = d2.loc[common]

delta = d2["suitability"] - d1["suitability"]
median = d1["skill_match_norm"].median()

print("mean Δ (high skill_match):", delta[d1["skill_match_norm"] >= median].mean())
print("mean Δ (low  skill_match):", delta[d1["skill_match_norm"] <  median].mean())


mean Δ (high skill_match): 0.0037232140256978578
mean Δ (low  skill_match): -0.007515279622804249


In [26]:
profile_r1, jobs_df_r1, gaps_df_r1, sensitivity_r1 = run_positioning(skill_text=skill_text,
                                                                    current_state='ALL',
                                                                    job_title_family='data_scientist',
                                                                    job_title_rich='ML_AI_data_scientist',
                                                                    target_sectors=None,
                                                                    w_skill= 0.7,
                                                                    salary_target=250000,
                                                                    explain_skills=False,
                                                                    top_k_gaps = 50,
                                                                    return_top_n_jobs = 200,
                                                                    run_sensitivity=False
                                                                    )

profile_r2, jobs_df_r2, gaps_df_r2, sensitivity_r2 = run_positioning(skill_text=skill_text,
                                                                    current_state='ALL',
                                                                    job_title_family='data_scientist',
                                                                    job_title_rich='ML_AI_data_scientist',
                                                                    target_sectors=None,
                                                                    w_skill= 0.7,
                                                                    salary_target=550000,
                                                                    explain_skills=False,
                                                                    top_k_gaps = 50,
                                                                    return_top_n_jobs = 200,
                                                                    run_sensitivity=False
                                                                    )

# align by job_id on the intersection
a = jobs_df_r1.set_index("job_id")["suitability"]
b = jobs_df_r2.set_index("job_id")["suitability"]

common = a.index.intersection(b.index)
a = a.loc[common]
b = b.loc[common]

# monotonic check: higher target => suitability should not increase
violations = (b > a + 1e-9).sum()
print("violations:", violations, "out of", len(common))

violations: 0 out of 62


In [27]:
jobs_df_r1

,job_id,Job Description,Rating,Size,Founded,Industry,Sector,role_source,state,ownership_clean,...,skill_PC9,skill_PC10,skill_match_score,skill_match_norm,salary_score,suitability,expected_missing,expected_missing_norm,salary_pct,competitiveness_index
2987,3006,Atlassian is continuing to hire with all inter...,4.4,1001 to 5000 employees,2002.0,Computer Hardware & Software,Information Technology,data_scientist,CA,public,...,0.162774,0.551393,0.559082,0.779541,0.900,0.815679,0.485500,0.017981,0.983871,0.500926
149,152,Fujitsu is the leading Japanese information an...,3.3,10000+ employees,1935.0,Unknown,Unknown,data_scientist,NY,public,...,0.069715,-0.366639,0.625061,0.812530,0.678,0.772171,0.545344,0.020198,0.838710,0.429454
2943,2962,Leading the future of luxury mobility\n\nLucid...,3.9,1001 to 5000 employees,2007.0,Transportation Equipment Manufacturing,Manufacturing,data_scientist,CA,private,...,0.303770,-0.735492,0.529161,0.764580,0.774,0.767406,0.152343,0.005642,0.935484,0.470563
2975,2994,"Job Description\nSeeking a high performing, an...",4.2,1 to 50 employees,2007.0,Enterprise Software & Network Solutions,Information Technology,data_scientist,CA,private,...,0.258877,0.070736,0.343254,0.671627,0.900,0.740139,0.312399,0.011570,0.983871,0.497721
3035,3055,Atlassian is continuing to hire with all inter...,4.4,1001 to 5000 employees,2002.0,Computer Hardware & Software,Information Technology,data_scientist,CA,public,...,0.162774,0.551393,0.559082,0.779541,0.614,0.729879,0.485500,0.017981,0.717742,0.367862
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3770,3795,"Job Description\nNew Challenge: New Team, New ...",3.6,201 to 500 employees,NaN,Oil & Gas Services,"Oil, Gas, Energy & Utilities",data_scientist,OH,private,...,0.028666,0.107696,-0.243510,0.378245,0.292,0.352371,0.790000,0.029259,0.112903,0.071081
1042,1049,"Do you have a background in Data Science, Phys...",2.7,1001 to 5000 employees,2019.0,TV Broadcast & Cable Networks,Media,data_scientist,IL,private,...,0.194701,-0.092692,-0.307243,0.346379,0.258,0.319865,0.517753,0.019176,0.088710,0.053943
1779,1789,ML Data Scientists ML Data Scientist shall pro...,4.3,1 to 50 employees,NaN,IT Services,Information Technology,data_scientist,PA,unknown,...,-0.046381,-0.190358,-0.324199,0.337901,0.258,0.313930,1.217894,0.045107,0.088710,0.066908
1339,1346,Job Description\nRole Description\n\nHands on ...,4.1,51 to 200 employees,1999.0,Enterprise Software & Network Solutions,Information Technology,data_scientist,AZ,private,...,-0.760708,-0.167597,-0.550967,0.224516,0.512,0.310761,0.222533,0.008242,0.451613,0.229927


#### competitiveness

In [28]:
profile_base, candidates_df_base, _, _ = run_positioning(skill_text='python, sql, coaching, cloud',
                                                                    current_state='ALL',
                                                                    job_title_family='data_scientist',
                                                                    job_title_rich='ML_AI_data_scientist',
                                                                    target_sectors=None,
                                                                    w_skill= 0.7,
                                                                    salary_target=250000,
                                                                    explain_skills=False,
                                                                    top_k_gaps = 50,
                                                                    return_top_n_jobs = 200,
                                                                    run_sensitivity=False
                                                                    )

profile_remove, _, _, _ = run_positioning(skill_text='python, cloud',
                                                                    current_state='ALL',
                                                                    job_title_family='data_scientist',
                                                                    job_title_rich='ML_AI_data_scientist',
                                                                    target_sectors=None,
                                                                    w_skill= 0.7,
                                                                    salary_target=250000,
                                                                    explain_skills=False,
                                                                    top_k_gaps = 50,
                                                                    return_top_n_jobs = 200,
                                                                    run_sensitivity=False
                                                                    )


profile_add, _, _, _ = run_positioning(skill_text='python, sql, coaching, cloud, git, gis',
                                                                    current_state='ALL',
                                                                    job_title_family='data_scientist',
                                                                    job_title_rich='ML_AI_data_scientist',
                                                                    target_sectors=None,
                                                                    w_skill= 0.7,
                                                                    salary_target=250000,
                                                                    explain_skills=False,
                                                                    top_k_gaps = 50,
                                                                    return_top_n_jobs = 200,
                                                                    run_sensitivity=False
                                                                    )

comp_base = add_competitiveness(profile_base, candidates_df_base, skill)
comp_remove = add_competitiveness(profile_remove, candidates_df_base, skill)
comp_add = add_competitiveness(profile_add, candidates_df_base, skill)

if (comp_add['job_id'] == comp_base['job_id']).mean() == 1 and (comp_remove['job_id'] == comp_base['job_id']).mean() == 1:
    print('✅ Identical index: PASS')
else:
    print('❌ Identical index: FAIL')  

if (comp_add['competitiveness_index'] <= comp_base['competitiveness_index']).mean() == 1:
    print('✅ Competitive skill behaviour add: PASS')
else:
    print('❌ Competitive skill behaviour add: FAIL')  

if (comp_remove['competitiveness_index'] >= comp_base['competitiveness_index']).mean() == 1:
    print('✅ Competitive skill behaviour remove: PASS')
else:
    print('❌ Competitive skill behaviour remove: FAIL')  

if (comp_add['expected_missing_norm'] <= comp_base['expected_missing_norm']).mean() == 1:
    print('✅ Missing skill behaviour add: PASS')
else:
    print('❌ Missing skill behaviour add: FAIL')  

if (comp_remove['expected_missing_norm'] >= comp_base['expected_missing_norm']).mean() == 1:
    print('✅ Missing skill behaviour remove: PASS')
else:
    print('❌ Missing skill behaviour remove: FAIL')  


✅ Identical index: PASS
✅ Competitive skill behaviour add: PASS
✅ Competitive skill behaviour remove: PASS
✅ Missing skill behaviour add: PASS
✅ Missing skill behaviour remove: PASS


### Sensitivity tests


In [29]:
profile_r1, jobs_df_r1, gaps_df_r1, sensitivity_r1 = run_positioning(skill_text=skill_text,
                                                                    current_state='ALL',
                                                                    job_title_family='data_scientist',
                                                                    job_title_rich='ML_AI_data_scientist',
                                                                    target_sectors=None,
                                                                    w_skill= 0.7,
                                                                    salary_target=250000,
                                                                    explain_skills=False,
                                                                    top_k_gaps = 50,
                                                                    return_top_n_jobs = 200,
                                                                    run_sensitivity=True
                                                                    )

sensitivity = sensitivity_r1
S = sensitivity["suitability"]
C = sensitivity["competitiveness"]

In [30]:
print("Sensitivity table:")
print("------------------")
print(S)
print("\n")
print("Competitiveness table:")
print("------------------")
print(C)
print("\n")




if max((S['w_skill'] + S['w_salary'] - 1).abs()) < 1e-9:
    print('✅ Suitability weight test: PASS')
else:
    print('❌ Suitability weight test: FAIL')  

if max((C['w_skill'] + C['w_salary'] - 1).abs()) < 1e-9:
    print('✅ Competitiveness weight test: PASS')
else:
    print('❌ Competitiveness weight test: FAIL')  

if S[S['w_skill'] == 0.7]['spearman_rho_vs_baseline'].iloc[0] == 1:
    print('✅ Suitability baseline test: PASS')
else:
    print('❌ Suitability baseline test: FAIL')

if C[C['w_skill'] == 0.7]['spearman_rho_vs_baseline'].iloc[0] == 1:
    print('✅ Competitiveness baseline test: PASS')
else:
    print('❌ Competitiveness baseline test: FAIL')

if len(S[(S['spearman_rho_vs_baseline'] >=0) & (S['spearman_rho_vs_baseline'] <=1)]) == len(S):
    print('✅ Suitability spearman range test: PASS')
else:
    print('❌ Suitability spearman range test: FAIL')

if len(C[(C['spearman_rho_vs_baseline'] >=0) & (C['spearman_rho_vs_baseline'] <=1)]) == len(C):
    print('✅ Competitiveness spearman range test: PASS')
else:
    print('❌ Competitiveness spearman range test: FAIL')

if ((S['spearman_rho_vs_baseline'][~((S['w_skill'] == 0.7) & (S['w_salary'] ==0.3))]) < 1 - 1e-9).any():
    print('✅ Suitability degenerate test: PASS')
else:
    print('❌ Suitability degenerate test: FAIL')

if ((C['spearman_rho_vs_baseline'][~((C['w_skill'] == 0.7) & (C['w_salary'] ==0.3))]) < 1 - 1e-9).any():
    print('✅ Competitiveness degenerate test: PASS')
else:
    print('❌ Competitiveness degenerate test: FAIL')



Sensitivity table:
------------------
   w_skill  w_salary  spearman_rho_vs_baseline
0      0.1       0.9                  0.630018
1      0.2       0.8                  0.688643
2      0.3       0.7                  0.760816
3      0.4       0.6                  0.847645
4      0.5       0.5                  0.922135
5      0.6       0.4                  0.973004
6      0.7       0.3                  1.000000
7      0.8       0.2                  0.967011
8      0.9       0.1                  0.901838


Competitiveness table:
------------------
   w_skill  w_salary  spearman_rho_vs_baseline
0      0.1       0.9                  0.971947
1      0.2       0.8                  0.977135
2      0.3       0.7                  0.983481
3      0.4       0.6                  0.989625
4      0.5       0.5                  0.994208
5      0.6       0.4                  0.997482
6      0.7       0.3                  1.000000
7      0.8       0.2                  0.995064
8      0.9       0.1     

# == End of Notebook ==